# PRIDE Metric Playground

**PRIDE** (Paraphrase Robustness Index in Robotic Instructional DEviation)

Interactive exploration of PRIDE components:
- **S_K** (Keyword Similarity): Content word preservation via Sentence-BERT cosine similarity
- **S_T** (Structural Similarity): Syntactic divergence via normalized tree edit distance
- **PD** (Paraphrase Distance): `PD = 1 - (alpha * S_K + (1 - alpha) * S_T)`
- **PRIDE**: `sum(success_i * PD_i) / sum(PD_i) * 100` (normalized 0-100)

In [ ]:

import spacy
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import networkx as nx

## 1. Enter two sentences

In [2]:
SENT_ORIGINAL = "open the middle drawer of the cabinet"
SENT_PARAPHRASE = "could you open the middle drawer of the cabinet"

nlp = spacy.load('en_core_web_sm')
doc_orig = nlp(SENT_ORIGINAL)
doc_para = nlp(SENT_PARAPHRASE)

NameError: name 'spacy' is not defined

## 2. Dependency tree side-by-side

S_T is computed from the tree edit distance between these dependency trees.

In [ ]:
def doc_to_tree(doc):
    """Convert spacy doc to networkx DiGraph."""
    G = nx.DiGraph()
    for tok in doc:
        label = f"{tok.text}\n({tok.dep_})"
        G.add_node(tok.i, label=label, dep=tok.dep_, pos=tok.pos_)
        if tok.head.i != tok.i:
            G.add_edge(tok.head.i, tok.i)
    return G

def hierarchy_pos(G, root=None, width=1., vert_gap=0.2, vert_loc=0, xcenter=0.5):
    """Position nodes in a top-down tree layout."""
    pos = _hierarchy_pos(G, root, width, vert_gap, vert_loc, xcenter)
    return pos

def _hierarchy_pos(G, root, width, vert_gap, vert_loc, xcenter, pos=None, parent=None, parsed=None):
    if pos is None:
        pos = {}
        parsed = set()
    pos[root] = (xcenter, vert_loc)
    parsed.add(root)
    children = [n for n in G.successors(root) if n not in parsed]
    if len(children) > 0:
        dx = width / len(children)
        nextx = xcenter - width / 2 - dx / 2
        for child in children:
            nextx += dx
            pos = _hierarchy_pos(G, child, width=dx, vert_gap=vert_gap,
                                 vert_loc=vert_loc - vert_gap, xcenter=nextx,
                                 pos=pos, parent=root, parsed=parsed)
    return pos

def find_root(doc):
    for tok in doc:
        if tok.head.i == tok.i:
            return tok.i
    return 0

# Dep relation color scheme (matching paper Fig.4)
DEP_COLORS = {
    'ROOT': '#E74C3C',    # red — sentence root
    'dobj': '#3498DB',    # blue — direct object
    'pobj': '#2ECC71',    # green — object of preposition
}
DEP_DEFAULT = '#95A5A6'   # gray — all other relations

def get_node_color(dep):
    return DEP_COLORS.get(dep, DEP_DEFAULT)

def draw_tree(ax, doc, title):
    """Draw dependency tree colored by dependency relation."""
    G = doc_to_tree(doc)
    root = find_root(doc)
    pos = hierarchy_pos(G, root=root, width=2., vert_gap=0.4)
    
    labels = nx.get_node_attributes(G, 'label')
    node_deps = nx.get_node_attributes(G, 'dep')
    colors = [get_node_color(node_deps.get(n, '')) for n in G.nodes()]
    
    nx.draw(G, pos, ax=ax, labels=labels, with_labels=True,
            node_color=colors, node_size=2000, font_size=7, font_weight='bold',
            arrows=True, arrowsize=15, arrowstyle='->', 
            edge_color='#444444', width=1.5,
            edgecolors='black', linewidths=0.8)
    ax.set_title(title, fontsize=11, fontweight='bold', pad=10)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
draw_tree(ax1, doc_orig, f'Original: "{SENT_ORIGINAL}"')
draw_tree(ax2, doc_para, f'Paraphrase: "{SENT_PARAPHRASE}"')

legend_items = [mpatches.Patch(color='#E74C3C', label='root (sentence root)'),
                mpatches.Patch(color='#3498DB', label='dobj (direct object)'),
                mpatches.Patch(color='#2ECC71', label='pobj (object of preposition)'),
                mpatches.Patch(color='#95A5A6', label='others')]
fig.legend(handles=legend_items, loc='lower center', ncol=4, fontsize=10, 
           frameon=True, edgecolor='#CCCCCC', fancybox=True)
plt.tight_layout(rect=[0, 0.07, 1, 1])
plt.show()

## 3. Token-level comparison

In [ ]:
print(f'{"Token":<15} {"POS":<8} {"DEP":<12} {"Head":<15}')
print('=' * 50)
print(f'--- Original: "{SENT_ORIGINAL}" ---')
for tok in doc_orig:
    print(f'{tok.text:<15} {tok.pos_:<8} {tok.dep_:<12} {tok.head.text:<15}')
print(f'\n--- Paraphrase: "{SENT_PARAPHRASE}" ---')
for tok in doc_para:
    print(f'{tok.text:<15} {tok.pos_:<8} {tok.dep_:<12} {tok.head.text:<15}')

print(f'\nTree depth (original):   {max(len(list(tok.ancestors)) for tok in doc_orig) + 1}')
print(f'Tree depth (paraphrase): {max(len(list(tok.ancestors)) for tok in doc_para) + 1}')

## 4. S_K (Keyword Similarity) & S_T (Structural Similarity)

- **S_K**: Content word (NOUN, VERB, ADJ, ADV) overlap — approximation of the Sentence-BERT matching used in the full pipeline
- **S_T**: Dependency relation pair overlap — approximation of the tree edit distance

In [ ]:
# S_K: content word overlap (approximation)
content_pos = {'NOUN', 'VERB', 'ADJ', 'ADV'}
kw_orig = set(tok.lemma_.lower() for tok in doc_orig if tok.pos_ in content_pos)
kw_para = set(tok.lemma_.lower() for tok in doc_para if tok.pos_ in content_pos)
s_k = len(kw_orig & kw_para) / max(len(kw_orig | kw_para), 1)

# S_T: dependency relation overlap (approximation)
deps_orig = set((tok.dep_, tok.head.dep_) for tok in doc_orig)
deps_para = set((tok.dep_, tok.head.dep_) for tok in doc_para)
s_t = len(deps_orig & deps_para) / max(len(deps_orig | deps_para), 1)

print(f'Content words (original):   {kw_orig}')
print(f'Content words (paraphrase): {kw_para}')
print(f'Shared: {kw_orig & kw_para}')
print(f'Added:  {kw_para - kw_orig}')
print()
print(f'S_K (Keyword Similarity):    {s_k:.3f}')
print(f'S_T (Structural Similarity): {s_t:.3f}')

## 5. Paraphrase Distance (PD) & PRIDE

```
PD_i = 1 - (alpha * S_K_i + (1 - alpha) * S_T_i)
PRIDE(alpha) = sum(success_i * PD_i) / sum(PD_i) * 100
```

PRIDE = 100 means all episodes succeeded, PRIDE = 0 means all failed. Harder paraphrases (higher PD) contribute more to the score.

In [ ]:
alphas = np.linspace(0, 1, 101)
pds = [1.0 - (a * s_k + (1 - a) * s_t) for a in alphas]

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(alphas, pds, linewidth=2)
ax.axvline(x=0.5, color='red', linestyle='--', alpha=0.5, label='alpha=0.5')

pd_st_only = 1.0 - s_t
pd_balanced = 1.0 - (0.5 * s_k + 0.5 * s_t)
pd_sk_only = 1.0 - s_k
ax.scatter([0, 0.5, 1], [pd_st_only, pd_balanced, pd_sk_only],
           s=80, c=['tab:orange', 'red', 'tab:blue'], zorder=5, edgecolors='black')

ax.set_xlabel('alpha (S_K weight)', fontweight='bold')
ax.set_ylabel('Paraphrase Distance (PD)', fontweight='bold')
ax.set_title('PD across alpha for this sentence pair', fontweight='bold')
ax.legend()
ax.grid(alpha=0.2)

ax.annotate(f'ST-only: PD={pd_st_only:.3f}', (0, pd_st_only), textcoords='offset points',
            xytext=(10, 10), fontsize=9)
ax.annotate(f'SK-only: PD={pd_sk_only:.3f}', (1, pd_sk_only), textcoords='offset points',
            xytext=(-100, 10), fontsize=9)
plt.tight_layout()
plt.show()

print(f'\nAt alpha=0.5:')
print(f'  S_K = {s_k:.3f}, S_T = {s_t:.3f}')
print(f'  PD  = {pd_balanced:.3f}')
print(f'  If this episode succeeds -> contributes PD={pd_balanced:.3f} to PRIDE numerator')
print(f'  If this episode fails    -> contributes 0 to PRIDE numerator')
print(f'  Always contributes PD={pd_balanced:.3f} to PRIDE denominator')